# Device Topology

Every warehouse row records the hardware and OS identity of the device that produced the measurement. These are the *topology* fields: they describe the machine, not the run. They are distinct from the runtime telemetry columns (thermal, battery, power state, cpuset/affinity), which change from run to run on the same machine.

This notebook walks through the twelve topology fields — what they are, which of them are optional, and how to group results by them.

Point it at a warehouse root (the directory containing `results/`) with the `PIPETTE_WAREHOUSE` environment variable, or run it from a checkout whose configured storage root is `./sample_data`:

```bash
PIPETTE_WAREHOUSE=/path/to/warehouse uv run jupyter notebook examples/notebooks
```

In [ ]:
import os
from pathlib import Path

import pandas as pd


def find_warehouse(start: Path) -> Path:
    configured = os.environ.get("PIPETTE_WAREHOUSE")
    if configured:
        warehouse = Path(configured)
        if not warehouse.exists():
            raise FileNotFoundError(f"PIPETTE_WAREHOUSE does not exist: {warehouse}")
        return warehouse.resolve()
    candidates = [
        start / "sample_data" / "warehouse",
        start.parent / "sample_data" / "warehouse",
        Path.cwd() / "sample_data" / "warehouse",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "No warehouse found. Set PIPETTE_WAREHOUSE to a warehouse root."
    )


WAREHOUSE = find_warehouse(Path.cwd())
WAREHOUSE

In [ ]:
# The topology fields, in the order they describe a machine: what it is, what
# silicon it runs on, how much memory it has, and what OS is installed.
TOPOLOGY_FIELDS = [
    "device_name",
    "device_form_factor",
    "device_chip_model",
    "device_gpu_model",
    "device_gpu_vram_bytes",
    "device_npu_model",
    "device_npu_vram_bytes",
    "device_ram_bytes",
    "device_os_name",
    "device_os_version",
    "device_os_build",
    "device_os_security_patch",
]


def load_results(base: Path) -> pd.DataFrame:
    paths = sorted((base / "results").rglob("*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No parquet files under {base / 'results'}")
    frame = pd.concat((pd.read_parquet(path) for path in paths), ignore_index=True)
    frame["submitted_at"] = pd.to_datetime(frame["submitted_at"], utc=True)
    return frame


df = load_results(WAREHOUSE)
AVAILABLE = [field for field in TOPOLOGY_FIELDS if field in df.columns]
print(f"{len(df):,} rows from {df['client_id'].nunique()} clients")

## 1. The fields

Each field, its Parquet dtype, and how many distinct values the warehouse holds for it. A field missing from `df.columns` means the data predates it — the schema is free to change, so treat presence as something to check rather than assume.

In [ ]:
rows = []
for field in TOPOLOGY_FIELDS:
    in_schema = field in df.columns
    rows.append(
        {
            "field": field,
            "present": in_schema,
            "dtype": str(df[field].dtype) if in_schema else "-",
            "distinct": int(df[field].nunique(dropna=True)) if in_schema else 0,
        }
    )

pd.DataFrame(rows)

## 2. Topology per client

A `client_id` is one machine, so its topology should collapse to a single row. More than one row for a client means the machine changed underneath it — an OS upgrade, a hardware change, or a client id reused across two machines.

In [ ]:
topology = (
    df[["client_id", *AVAILABLE]]
    .drop_duplicates()
    .sort_values("client_id")
    .reset_index(drop=True)
)
topology

## 3. Which fields are optional

Six fields are populated on every row: `device_name`, `device_form_factor`, `device_chip_model`, `device_ram_bytes`, `device_os_name`, and `device_os_version`. The rest are null whenever the platform has nothing to report — a CPU-only box has no GPU or NPU, and only Android reports an OS build and a security patch level. Check coverage before filtering on any of them.

In [ ]:
coverage = (
    df[AVAILABLE]
    .notna()
    .mean()
    .mul(100)
    .round(1)
    .rename("populated_pct")
    .to_frame()
    .sort_values("populated_pct")
)
coverage

## 4. Grouping results by topology

The point of the fields: comparing measurements across hardware classes rather than across opaque client ids. Byte counts are stored raw, so convert them for display.

In [ ]:
by_topology = (
    df.assign(ram_gib=(df["device_ram_bytes"] / 1024**3).round(1))
    .groupby(["device_form_factor", "device_os_name", "device_name", "ram_gib"])
    .agg(clients=("client_id", "nunique"), results=("result_id", "size"))
    .reset_index()
    .sort_values(["device_form_factor", "device_name"])
)
by_topology

## 5. Topology drift over time

Because the topology travels on every row rather than living in a separate registry, history is preserved: an OS upgrade shows up as a second `device_os_version` for the same client, with the changeover visible in the timestamps. One row per client means nothing moved. Two rows is the signal that results either side of that boundary are not strictly comparable.

In [ ]:
os_history = (
    df.groupby(["client_id", "device_os_name", "device_os_version"])
    .agg(first_seen=("submitted_at", "min"), last_seen=("submitted_at", "max"))
    .reset_index()
    .sort_values(["client_id", "first_seen"])
    .reset_index(drop=True)
)
os_history